# Solution E2 · A tree with AUC 0.99 on loans: it was the identifier

**Type**: Find the bug

## Objective

Find out why a tree on the loans reaches AUC 0.99 in cross-validation: the dataset includes `customer_id`, assigned in order of **file closing** (defaulted loans have higher ids), so a threshold on the id separates almost perfectly. Measure the AUC on **new customers** (ids even higher) and
remove the column.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier


def dataset():
    df = ds.loans(n=1800, with_missing=False)
    df["ratio"] = df["amount"] / df["income"]
    df = df.sample(frac=1, random_state=0).reset_index(drop=True)
    train, new = df.iloc[:1500].copy(), df.iloc[1500:].copy()
    train = train.sort_values("defaulted", kind="stable").reset_index(drop=True)      # ids in order of outcome: first the regular ones, then the defaulters
    train["customer_id"] = np.arange(1500)
    new["customer_id"] = np.arange(1500, 1800)                                         # the new customers have ids above the maximum of the training set
    return train, new


train, new = dataset()
FEATURES = ["ratio", "age", "job_tenure", "income", "amount"]
X_tr, y_tr = train[FEATURES + ["customer_id"]], train["defaulted"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("defaulters in the training set:", round(y_tr.mean(), 3), "| ids of the new customers:", int(new["customer_id"].min()), "-", int(new["customer_id"].max()))

## Commented solution

In [ ]:
tree = DecisionTreeClassifier(min_samples_leaf=30, random_state=0)
cv_auc_bug = cross_val_score(tree, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
id_importance = pd.Series(tree.fit(X_tr, y_tr).feature_importances_, index=X_tr.columns)["customer_id"]
print(f"cross-validation AUC: {cv_auc_bug:.3f} | importance of customer_id: {id_importance:.2f}")

**Where the bug is.** `customer_id` was assigned **after** the outcome (in order of closing): the regular customers have low ids and the defaulters high ids. A single threshold on the id separates the classes almost perfectly, and a tree finds it at the root. In cross-validation it works (the validation ids are in the same range as the training ones),
but for a new customer the id is simply **higher than all the others**: it falls in the last leaf, the one of the defaulters (or of the regular ones), and all the new customers get the same score.

In [ ]:
X_new, y_new = new[FEATURES + ["customer_id"]], new["defaulted"]
id_outcome_corr = float(spearmanr(train["customer_id"], train["defaulted"])[0])
with_id = DecisionTreeClassifier(min_samples_leaf=30, random_state=0).fit(X_tr, y_tr)
p_new = with_id.predict_proba(X_new)[:, 1]
new_auc_bug = roc_auc_score(y_new, p_new)
n_distinct_new_values = int(np.unique(p_new).size)
print(f"id-outcome correlation: {id_outcome_corr:.2f} | AUC on the new customers: {new_auc_bug:.3f} | distinct values of predict_proba on the new ones: {n_distinct_new_values}")

In [ ]:
without_id = DecisionTreeClassifier(min_samples_leaf=30, random_state=0)
cv_auc_ok = cross_val_score(without_id, X_tr[FEATURES], y_tr, cv=cv, scoring="roc_auc").mean()
new_auc_ok = roc_auc_score(y_new, without_id.fit(X_tr[FEATURES], y_tr).predict_proba(X_new[FEATURES])[:, 1])
print(f"without customer_id: cross-validation AUC {cv_auc_ok:.3f} | on the new customers {new_auc_ok:.3f}")

## Check

In [ ]:
with verify():
    check(cv_auc_bug > 0.95 and id_importance > 0.8, "the colleague: AUC over 0.95 and customer_id is worth over 80% of the importance", "don't modify the colleague's code")
    check(id_outcome_corr is not None and id_outcome_corr > 0.6, "the id is strongly correlated with the outcome (Spearman over 0.6)", "spearmanr(train['customer_id'], train['defaulted'])")
    check(new_auc_bug is not None and 0.45 < new_auc_bug < 0.55 and n_distinct_new_values <= 2, "on the new customers the AUC is 0.5: the model with the id gives everyone the same score (at most two distinct values)", "all the new ids are beyond the last threshold")
    check(cv_auc_ok is not None and 0.65 < cv_auc_ok < 0.85 and new_auc_ok > 0.62, "without the id: cross-validation AUC 0.65-0.85, on the new customers over 0.62", "FEATURES only (without customer_id)")
done()

## What you learned

- **An AUC of 0.99 on a credit problem is a warning sign**, not a success: the realistic value is about 0.73-0.78 (M4). When a model is "too good", look first at the **importances**: `customer_id` was worth 100%.
- **An identifier that encodes the order of collection is not a feature.** Here the id is assigned after the outcome; in other cases it is a timestamp, a batch number, the file name. Cross-validation does not see it (the validation folds are in the same id range as the training set) and in production the id is out of range.
- **The new customers all receive the same score** (the AUC is 0.5, `predict_proba` has at most two values): the tree has thresholds on the id only inside the range seen, and an id higher than all the others falls in the last leaf. A useless model with a validation that said 0.99.
- Without the id the tree has AUC 0.73 (in cross-validation and on the new customers), consistent with what other models get. Removing suspicious columns (ids, dates, codes) is a standard step of a serious pipeline: list them explicitly in a `ColumnTransformer` with `remainder="drop"`.

## Common pitfall

Leaving the "technical" columns (ids, row indices, insertion timestamps) in the dataset because "the model will work out by itself whether they are needed". A tree uses them if they correlate with the target in the sample, even for a reason that does not hold in production.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Your model has a very high AUC in validation. What do you check before trusting it?</summary>

The feature importances (a dominant column, especially if it is an id or a timestamp), the structure of the validation (same period/same customers in training and test), consistency with a simple baseline, and I run a test with new data obtained the way it would be in production. I remove identifiers and columns that
encode the collection process, and repeat the evaluation with a temporal or group split.

</details>